# Variabilidad hospitalaria en la práctica clínica

Proyecto aplicado (Chile, 2019–2024) usando los egresos reales de `DATASET-PROBLEMA8/`.

**Objetivo en una frase:** aislar la variabilidad atribuible al hospital cuando los pacientes tienen diagnóstico y severidad comparables.

### Cómo usar este notebook (2 min de lectura)
- Ejecuta las celdas en orden. Si falta algún CSV, la celda de carga te avisará cuál.
- Todo lo teórico va en Word; aquí dejamos separadores con instrucciones claras.
- El flujo ya está limpio y comentado: carga → limpieza → GRD focal → descriptiva → inferencial.
- Ajusta `target_grd` o los umbrales si necesitas otro GRD o cortes diferentes.

## 1.1 Introducción

> **[PARA WORD | APA 7]**
> - Contexto del problema y por qué importa a pacientes y gestores.
> - Qué significa “variabilidad” en la práctica clínica.
> - Brecha que se busca cerrar con este análisis.

## 1.2 Revisión Bibliográfica

> **[PARA WORD | APA 7]**
> - Evidencia previa sobre variabilidad hospitalaria y GRD.
> - Cómo otros estudios ajustan por severidad/riesgo.
> - Hallazgos clave que justifican mirar Chile 2019–2024.

## 1.3 Pregunta de Investigación

> **[PARA WORD | APA 7]**
> ¿En qué medida el hospital determina los días de estada y la intensidad de procedimientos en pacientes clínicamente comparables?

## 1.4 Hipótesis

> **[PARA WORD | APA 7]**
> Existe heterogeneidad estadísticamente significativa en resultados asistenciales entre hospitales, incluso manteniendo constante la severidad.

### Ruta del análisis en 5 pasos
1) Importar librerías y estilos.
2) Cargar y concatenar los CSV 2019–2024.
3) Detectar columnas clave y limpiar (nulos + outliers p99).
4) Fijar un GRD de referencia, describir y visualizar.
5) Probar hipótesis (normalidad + Kruskal-Wallis) y dejar conclusiones para Word.

In [1]:
# ==========================================
# FASE 2: Metodología y Preparación de Datos
# ==========================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", context="talk")

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


### Carga y concatenación de los CSV
Los seis archivos del directorio `DATASET-PROBLEMA8/` se unen en un solo DataFrame, conservando el nombre de origen para trazabilidad.

In [ ]:
# Carga directa de los CSV con rutas literales (sin bucles)
# Convierte cada archivo a UTF-8 antes de leerlo.

def convert_csv_to_utf8(path):
    src = Path(path)
    if not src.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    utf8_dir = src.parent / "utf8"
    utf8_dir.mkdir(parents=True, exist_ok=True)
    dst = utf8_dir / src.name

    raw = src.read_bytes()
    text = None
    used_encoding = None

    for enc in ["utf-8-sig", "utf-8", "utf-16", "utf-16-le", "latin-1", "cp1252"]:
        try:
            text = raw.decode(enc)
            used_encoding = enc
            break
        except UnicodeDecodeError:
            continue

    if text is None:
        # Último recurso para no bloquear el flujo por bytes corruptos
        text = raw.decode("utf-8", errors="replace")
        used_encoding = "utf-8 (errors=replace)"

    dst.write_text(text, encoding="utf-8", newline="")
    print(f"Convertido a UTF-8: {src.name} (origen detectado: {used_encoding})")
    return dst


def read_hosp_csv(path):
    try:
        utf8_path = convert_csv_to_utf8(path)
        df = pd.read_csv(
            utf8_path,
            sep="|",
            engine="python",
            on_bad_lines="skip",
            encoding="utf-8"
        )
        return df
    except Exception as e:
        print(f"Error cargando {path}: {e}")
        return pd.DataFrame()


df_2019 = read_hosp_csv("DATASET-PROBLEMA8/GRD_PUBLICO_2019.csv")
df_2020 = read_hosp_csv("DATASET-PROBLEMA8/GRD_PUBLICO_2020.csv")
df_2021 = read_hosp_csv("DATASET-PROBLEMA8/GRD_PUBLICO_2021.csv")
df_2022 = read_hosp_csv("DATASET-PROBLEMA8/GRD_PUBLICO_EXTERNO_2022.csv")
df_2023 = read_hosp_csv("DATASET-PROBLEMA8/GRD_PUBLICO_2023.csv")
df_2024 = read_hosp_csv("DATASET-PROBLEMA8/GRD_PUBLICO_2024.csv")

dfs = [
    df_2019.assign(archivo_origen="GRD_PUBLICO_2019.csv"),
    df_2020.assign(archivo_origen="GRD_PUBLICO_2020.csv"),
    df_2021.assign(archivo_origen="GRD_PUBLICO_2021.csv"),
    df_2022.assign(archivo_origen="GRD_PUBLICO_EXTERNO_2022.csv"),
    df_2023.assign(archivo_origen="GRD_PUBLICO_2023.csv"),
    df_2024.assign(archivo_origen="GRD_PUBLICO_2024.csv"),
]

# Solo dataframes no vacíos
dfs = [d for d in dfs if not d.empty]

if not dfs:
    raise ValueError("No se pudo cargar ningún archivo. Revisa ruta y formato CSV.")

df_egresos = pd.concat(dfs, axis=0, ignore_index=True)

print(f"Archivos cargados: {len(dfs)}")
print(f"Registros totales: {len(df_egresos):,}")
print(f"Columnas detectadas: {len(df_egresos.columns)}")
display(df_egresos.head(3))

Convertido a UTF-8: GRD_PUBLICO_2019.csv (origen detectado: utf-8-sig)
Convertido a UTF-8: GRD_PUBLICO_2020.csv (origen detectado: utf-8-sig)
Convertido a UTF-8: GRD_PUBLICO_2021.csv (origen detectado: utf-8-sig)
Convertido a UTF-8: GRD_PUBLICO_EXTERNO_2022.csv (origen detectado: utf-16)
Convertido a UTF-8: GRD_PUBLICO_2023.csv (origen detectado: utf-16)
Convertido a UTF-8: GRD_PUBLICO_2024.csv (origen detectado: latin-1)


In [1]:
# Estandarización inteligente de variables críticas según nombres reales detectados

def choose_column(df, candidates, label):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for c in candidates:
        if c.lower().strip() in lower_map:
            return lower_map[c.lower().strip()]
    raise KeyError(f"No se encontró columna para '{label}'. Candidatas: {candidates}")

hospital_col = choose_column(df_egresos, [
    "codigo_establecimiento", "cod_establecimiento", "establecimiento", "hospital", "cod_hospital"
], "hospital")

dias_col = choose_column(df_egresos, [
    "dias_estada", "estada_dias", "diasestad", "dia_estada", "dias de estada"
], "dias_estada")

grd_col = choose_column(df_egresos, [
    "codigo_grd", "cod_grd", "grd", "grd_codigo"
], "codigo_grd")

cie10_col = choose_column(df_egresos, [
    "diagnostico_principal", "cie10_principal", "codigo_cie10", "cod_cie10", "diag_principal"
], "codigo_cie10")

proc_count_candidates = ["cantidad_procedimientos", "n_procedimientos", "num_procedimientos", "total_procedimientos"]
proc_code_candidates = ["codigo_cie9", "cod_cie9", "proc_cie9", "procedimiento_codigo"]

proc_count_col = None
for c in proc_count_candidates:
    if c.lower() in [x.lower() for x in df_egresos.columns]:
        proc_count_col = [x for x in df_egresos.columns if x.lower() == c.lower()][0]
        break

proc_code_col = None
for c in proc_code_candidates:
    if c.lower() in [x.lower() for x in df_egresos.columns]:
        proc_code_col = [x for x in df_egresos.columns if x.lower() == c.lower()][0]
        break

df = df_egresos[[hospital_col, dias_col, grd_col, cie10_col]].copy()
df.columns = ["hospital", "dias_estada", "codigo_grd", "codigo_cie10"]

if proc_count_col is not None:
    df["cantidad_procedimientos"] = pd.to_numeric(df_egresos[proc_count_col], errors="coerce")
    proc_mode = "conteo_directo"
elif proc_code_col is not None:
    df["cantidad_procedimientos"] = (~df_egresos[proc_code_col].isna()).astype(int)
    proc_mode = "proxy_por_codigo_cie9"
else:
    df["cantidad_procedimientos"] = np.nan
    proc_mode = "sin_variable_disponible"

print("Mapeo de columnas utilizado:")
print({
    "hospital": hospital_col,
    "dias_estada": dias_col,
    "codigo_grd": grd_col,
    "codigo_cie10": cie10_col,
    "procedimientos_modo": proc_mode,
    "procedimientos_columna": proc_count_col if proc_count_col is not None else proc_code_col
})

NameError: name 'df_egresos' is not defined

In [ ]:
# Limpieza: nulos críticos y outliers (percentil 99 de días de estada)

n0 = len(df)

df["dias_estada"] = pd.to_numeric(df["dias_estada"], errors="coerce")
df["codigo_grd"] = df["codigo_grd"].astype(str)
df["hospital"] = df["hospital"].astype(str)

df = df.dropna(subset=["hospital", "dias_estada", "codigo_grd"]).copy()
df = df[df["dias_estada"] >= 0].copy()

p99 = df["dias_estada"].quantile(0.99)
df = df[df["dias_estada"] <= p99].copy()

n1 = len(df)
print(f"Registros antes: {n0:,}")
print(f"Registros después: {n1:,}")
print(f"Eliminados: {n0 - n1:,}")
print(f"Corte outlier p99 (días de estada): {p99:.2f}")

## FASE 3: Análisis Exploratorio de Datos y Descriptiva
Radiografía rápida del GRD elegido: cuántos casos por hospital, cómo se distribuyen los días de estada y cuántos procedimientos se hacen en promedio.

In [ ]:
# Selección de GRD de alta prevalencia

target_grd = "GRD_89"
grd_series = df["codigo_grd"].astype(str)

if target_grd in set(grd_series.unique()):
    selected_grd = target_grd
else:
    selected_grd = grd_series.value_counts().index[0]

df_focus = df[grd_series == selected_grd].copy()

print(f"GRD seleccionado: {selected_grd}")
print(f"Registros: {len(df_focus):,}")
print(f"Hospitales únicos: {df_focus['hospital'].nunique()}")

In [ ]:
# Estadística descriptiva por hospital

summary = (
    df_focus.groupby("hospital", dropna=False)
    .agg(
        count=("dias_estada", "count"),
        dias_mean=("dias_estada", "mean"),
        dias_median=("dias_estada", "median"),
        dias_std=("dias_estada", "std"),
        dias_var=("dias_estada", "var"),
        proc_mean=("cantidad_procedimientos", "mean"),
        proc_median=("cantidad_procedimientos", "median"),
        proc_std=("cantidad_procedimientos", "std"),
        proc_var=("cantidad_procedimientos", "var")
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

summary.head(25)

In [ ]:
# Visualización 1: Boxplot de días de estada por hospital

top_hosp = df_focus["hospital"].value_counts().head(15).index
df_plot = df_focus[df_focus["hospital"].isin(top_hosp)].copy()

plt.figure(figsize=(18, 8))
sns.boxplot(
    data=df_plot,
    x="hospital",
    y="dias_estada",
    showfliers=False,
    palette="viridis"
)
plt.title("Distribución de Días de Estada por Hospital (GRD Controlado)")
plt.xlabel("Hospital")
plt.ylabel("Días de estada")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: Barplot de promedio de procedimientos por hospital

avg_proc = (
    df_plot.groupby("hospital", as_index=False)["cantidad_procedimientos"]
    .mean()
    .sort_values("cantidad_procedimientos", ascending=False)
)

plt.figure(figsize=(18, 8))
sns.barplot(
    data=avg_proc,
    x="hospital",
    y="cantidad_procedimientos",
    palette="magma"
)
plt.title("Promedio de Procedimientos por Hospital (GRD Controlado)")
plt.xlabel("Hospital")
plt.ylabel("Promedio de procedimientos")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## FASE 4: Análisis Inferencial y Multivariado
Probamos si las diferencias entre hospitales son estadísticamente significativas: primero validamos que la distribución no es normal (esperable), luego aplicamos Kruskal-Wallis.

In [ ]:
# Normalidad (Shapiro-Wilk)

alpha = 0.05
x = df_focus["dias_estada"].dropna().values

if len(x) == 0:
    raise ValueError("No hay datos en 'dias_estada' para evaluar normalidad.")

if len(x) > 5000:
    rng = np.random.default_rng(42)
    x_norm = rng.choice(x, size=5000, replace=False)
    note = "(submuestra aleatoria n=5000)"
else:
    x_norm = x
    note = "(muestra completa)"

w_stat, p_norm = stats.shapiro(x_norm)

print("Shapiro-Wilk", note)
print(f"Estadístico W: {w_stat:.6f}")
print(f"P-value: {p_norm:.6e}")

if p_norm < alpha:
    print("Resultado: Se rechaza H0 de normalidad. La variable presenta asimetría/no normalidad.")
else:
    print("Resultado: No se rechaza H0 de normalidad.")

In [ ]:
# Kruskal-Wallis entre hospitales

min_cases = 20
valid_h = df_focus["hospital"].value_counts()
valid_h = valid_h[valid_h >= min_cases].index

df_kw = df_focus[df_focus["hospital"].isin(valid_h)].copy()

groups = [
    g["dias_estada"].dropna().values
    for _, g in df_kw.groupby("hospital")
]

if len(groups) < 2:
    raise ValueError("No hay suficientes hospitales para Kruskal-Wallis. Reduce 'min_cases'.")

h_stat, p_kw = stats.kruskal(*groups)

print("Prueba de Kruskal-Wallis")
print(f"Hospitales analizados: {len(groups)}")
print(f"Estadístico H: {h_stat:.6f}")
print(f"P-value: {p_kw:.6e}")

if p_kw < alpha:
    print("Conclusión (alpha=0.05): Se rechaza H0; existen diferencias significativas entre hospitales.")
else:
    print("Conclusión (alpha=0.05): No se rechaza H0; no hay evidencia suficiente de diferencias.")

## 5.1 Interpretación de Resultados

> **[INSTRUCCIÓN PARA EL EQUIPO: Redactar en Word.]**
>
> Interpretar p-values y resultados desde el punto de vista clínico y de gestión.

## 5.2 Limitaciones

> **[INSTRUCCIÓN PARA EL EQUIPO: Redactar en Word.]**
>
> Incluir sesgos de registro, calidad del dato y variables omitidas.

## 5.3 Futuras Líneas de Investigación

> **[INSTRUCCIÓN PARA EL EQUIPO: Redactar en Word.]**
>
> Proponer modelos avanzados, análisis por subgrupos y evaluación longitudinal.

### Qué llevarte a Word (recordatorio rápido)
- P-values y conclusiones de Kruskal-Wallis.
- Lectura clínica de por qué el hospital podría influir aun con severidad comparable.
- Limitaciones de dato y próximos pasos (modelos multivariados, análisis por especialidad).